# Conditional probability with neighbor pooling

Compare county × landcover color distributions with an inclusive county-neighbor pool.

## Setup
Imports, input paths, and the warning filter are kept together for the recorded environment.

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

## Step 1: Load the dataset

In [11]:
DATA_PATH = Path('../../../dataset/Capstone2025_nsi_lvl9_with_landcover_and_color.csv')
df = pd.read_csv(DATA_PATH, low_memory=False)

## Step 2: Create the county adjacency mapping

Each county's pool includes itself plus counties that share a border: $\text{pool}(i) = \{i\} \cup \{\text{adjacent counties}\}$. For Tulare County, this includes Kern, Kings, Fresno, and Inyo. Pooling increases sample size and accounts for spatial similarity.

In [12]:
NEIGHBORS_PATH = Path('../../../website/backend/data/ca_county_neighbors.csv')
neighbors_df = pd.read_csv(NEIGHBORS_PATH)

adjacency_map = {}
all_counties = set(neighbors_df['county_fips'].unique()) | set(neighbors_df['neighbor_fips'].unique())

for county in all_counties:
    pool = {county}
    pool.update(neighbors_df[neighbors_df['county_fips'] == county]['neighbor_fips'].tolist())
    pool.update(neighbors_df[neighbors_df['neighbor_fips'] == county]['county_fips'].tolist())
    adjacency_map[county] = sorted(list(pool))

## Step 3: Aggregate to county × landcover × color

Aggregate counts: $y_{i,j,k}$ = structures for county $i$, landcover $j$, color $k$. Exposure $n_{i,j} = \sum_k y_{i,j,k}$ is total structures per county×landcover. Observed proportion $p_{i,j,k}^{\text{county}} = y_{i,j,k} / n_{i,j}$.

In [13]:
county_lc_color_counts = df.groupby(['fips', 'lc_type', 'clr'])['clr_cc'].sum().reset_index()
county_lc_color_counts.rename(columns={'clr_cc': 'y_county'}, inplace=True)

county_exposure = county_lc_color_counts.groupby(['fips', 'lc_type'])['y_county'].sum().reset_index()
county_exposure.rename(columns={'y_county': 'n_county'}, inplace=True)

county_lc_color_counts = county_lc_color_counts.merge(county_exposure, on=['fips', 'lc_type'])
county_lc_color_counts['p_county'] = county_lc_color_counts['y_county'] / county_lc_color_counts['n_county']

## Step 4: Build neighbor-pool counts

For each county $i$, pool counts across $\text{pool}(i)$: $y_{i,j,k}^{\text{pool}} = \sum_{c \in \text{pool}(i)} y_{c,j,k}$ and $n_{i,j}^{\text{pool}} = \sum_{c \in \text{pool}(i)} n_{c,j}$. Pooled proportion $p_{i,j,k}^{\text{pool}} = y_{i,j,k}^{\text{pool}} / n_{i,j}^{\text{pool}}$ represents the regional distribution, accounting for spatial autocorrelation.

In [14]:
pooled_data = []
for county_fips, neighbor_pool in adjacency_map.items():
    pool_df = county_lc_color_counts[county_lc_color_counts['fips'].isin(neighbor_pool)].copy()
    pool_agg = pool_df.groupby(['lc_type', 'clr'])['y_county'].sum().reset_index()
    pool_agg.rename(columns={'y_county': 'y_pool'}, inplace=True)
    pool_exposure = pool_agg.groupby('lc_type')['y_pool'].sum().reset_index()
    pool_exposure.rename(columns={'y_pool': 'n_pool'}, inplace=True)
    pool_agg = pool_agg.merge(pool_exposure, on='lc_type')
    pool_agg['p_pool'] = pool_agg['y_pool'] / pool_agg['n_pool']
    pool_agg['fips'] = county_fips
    pool_agg['num_neighbors'] = len(neighbor_pool) - 1
    pooled_data.append(pool_agg)

neighborpool_lc_color_counts = pd.concat(pooled_data, ignore_index=True)

## Step 5: Apply smoothing

Apply Dirichlet prior with $\alpha = 1$ to avoid infinite surprisal: $p_{i,j,k} = (y_{i,j,k} + \alpha) / (n_{i,j} + \alpha \cdot K)$ where $K$ is the number of colors. This prevents zero probabilities while having minimal effect when exposure is large.

In [15]:
alpha = 1.0
K = df['clr'].nunique()

merged = county_lc_color_counts.merge(
    neighborpool_lc_color_counts[['fips', 'lc_type', 'clr', 'y_pool', 'n_pool', 'p_pool', 'num_neighbors']],
    on=['fips', 'lc_type', 'clr'],
    how='left'
)

merged['y_pool'] = merged['y_pool'].fillna(0)
merged['n_pool'] = merged['n_pool'].fillna(merged['n_county'])
merged['p_pool'] = merged['p_pool'].fillna(merged['p_county'])
merged['num_neighbors'] = merged['num_neighbors'].fillna(0)

merged['p_county'] = (merged['y_county'] + alpha) / (merged['n_county'] + alpha * K)
merged['p_pool'] = (merged['y_pool'] + alpha) / (merged['n_pool'] + alpha * K)

## Step 6: Compute anomaly scores

KL divergence: $\text{KL}_{i,j} = \sum_k p_{i,j,k}^{\text{county}} \log(p_{i,j,k}^{\text{county}} / p_{i,j,k}^{\text{pool}})$ measures information-theoretic difference. L1 distance: $\text{L1}_{i,j} = \frac{1}{2} \sum_k |p_{i,j,k}^{\text{county}} - p_{i,j,k}^{\text{pool}}|$ is more intuitive. Top contributing color is $\arg\max_k \text{contrib}_{i,j,k}$ where $\text{contrib}_{i,j,k} = p_{i,j,k}^{\text{county}} \log(p_{i,j,k}^{\text{county}} / p_{i,j,k}^{\text{pool}})$.

In [16]:
merged['contrib'] = merged['p_county'] * np.log(merged['p_county'] / merged['p_pool'])
merged['abs_diff'] = np.abs(merged['p_county'] - merged['p_pool'])
merged['contrib'] = merged['contrib'].replace([np.inf, -np.inf, np.nan], 0)

county_lc_summary = merged.groupby(['fips', 'lc_type']).agg({
    'n_county': 'first',
    'n_pool': 'first',
    'num_neighbors': 'first',
    'contrib': 'sum',
    'abs_diff': lambda x: 0.5 * x.sum()
}).reset_index()

county_lc_summary.rename(columns={'contrib': 'kl_div', 'abs_diff': 'l1_distance'}, inplace=True)

top_color_data = merged.loc[merged.groupby(['fips', 'lc_type'])['contrib'].idxmax()][
    ['fips', 'lc_type', 'clr', 'contrib']
].rename(columns={'clr': 'top_color', 'contrib': 'top_contrib'})

county_lc_summary = county_lc_summary.merge(top_color_data, on=['fips', 'lc_type'], how='left')

In [17]:
output_dir = Path('../../../results/tables/conditional_probability')
output_dir.mkdir(parents=True, exist_ok=True)

summary_cols = ['fips', 'lc_type', 'n_county', 'n_pool', 'num_neighbors', 
                'kl_div', 'l1_distance', 'top_color', 'top_contrib']
summary_export = county_lc_summary[summary_cols].copy()
summary_export.to_csv(output_dir / 'm01_neighbor_pool_county_lc_summary.csv', index=False)

detail_cols = ['fips', 'lc_type', 'clr', 'y_county', 'y_pool', 
               'p_county', 'p_pool', 'contrib', 'abs_diff']
detail_export = merged[detail_cols].copy()
detail_export.to_csv(output_dir / 'm01_neighbor_pool_county_lc_color_detail.csv', index=False)

## Outputs and handoff

Use the county × landcover summary and color-detail CSV exports for downstream anomaly review; their schemas are preserved here.